# Multimodal Pipeline subgraph end-to-end

> Notebook generated from [`examples/multimodal_pipeline.py`](../../examples/multimodal_pipeline.py).

| Field | Value |
|------|-------|
| **Dataset** | inline PNG bytes |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Walks router → vision → fusion → output_formatter. Prints router decision and final answer.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
End-to-end multimodal pipeline demo with mocked providers (Fase F).

Builds the multimodal subgraph, injects a fake vision agent (no VLM / network),
and runs an image through router → vision_node → fusion_node →
output_formatter_node.

Run::

    python examples/multimodal_pipeline.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
from unittest.mock import AsyncMock

from langchain_core.messages import HumanMessage

from prismal.agents.multimodal import VisionResult
from prismal.agents.subgraphs.multimodal_pipeline import build_multimodal_subgraph

# 1×1 transparent PNG-ish header (enough for MediaValidator magic-byte sniffing).
PNG = b"\x89PNG\r\n\x1a\n" + b"\x00" * 64


def _fake_vision_agent() -> AsyncMock:
    """A VisionAgent stand-in that returns a fixed caption (no VLM call)."""
    agent = AsyncMock()
    agent.analyze = AsyncMock(
        return_value=VisionResult(
            description="A golden retriever running on a sandy beach.",
            objects=[],
            ocr_text=None,
            model_used="mock-vlm",
        )
    )
    return agent


async def main() -> None:
    definition = build_multimodal_subgraph(
        vision_agent=_fake_vision_agent(),
        fusion_strategy="concat",  # deterministic, no LLM
    )

    # Caller supplies the media under metadata.mm.media and an attachment hint.
    state: dict = {
        "messages": [
            HumanMessage(
                content="What is in this image?",
                additional_kwargs={"attachments": [{"mime_type": "image/png"}]},
            )
        ],
        "metadata": {"mm": {"media": PNG, "preferred_output": "text"}},
    }

    # Drive the nodes the way the compiled subgraph would: router → modal → fusion → output.
    state.update(await definition.nodes["router"](state))
    next_node = definition.conditional_edges["router"](state)
    print("router → ", next_node)

    state["metadata"].update((await definition.nodes[next_node](state))["metadata"])
    state["metadata"].update((await definition.nodes["fusion_node"](state))["metadata"])
    out = await definition.nodes["output_formatter_node"](state)

    print("final answer:", out["messages"][-1].content)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()